# Validation Features Walkthrough

This notebook demonstrates BattINFO's validation layers with both valid and invalid examples.

It covers:

- profile/schema validation
- canonical record validation
- referential integrity validation
- semantic/domain validation
- publication JSON-LD validation


In [ ]:
import copy
import json
import tempfile
from pathlib import Path

from battinfo import CellInstance, CellType, Dataset, Test, publish
from battinfo.bundle import ProvenanceInfo
from battinfo.validate import (
    validate_json_report,
    validate_publication_report,
    validate_record_report,
    validate_references_report,
    validate_semantic_report,
)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

EXAMPLES_ROOT = REPO_ROOT / "src" / "battinfo" / "data" / "examples"
print("Repo root:", REPO_ROOT)
print("Examples root:", EXAMPLES_ROOT)


In [ ]:
def load_example(*parts: str) -> dict:
    path = EXAMPLES_ROOT.joinpath(*parts)
    return json.loads(path.read_text(encoding="utf-8"))


def show_report(label: str, report) -> None:
    print(f"\n=== {label} ===")
    print("ok:", report.ok)
    print("policy:", report.policy.name)
    if not report.issues:
        print("issues: none")
        return
    for issue in report.issues:
        print(f"- [{issue.severity}] {issue.code} | path={issue.path or '<root>'}")
        print(f"  {issue.message}")


## 1. Profile / Schema Validation

The profile validator is useful for non-canonical inputs such as manifests and descriptor-style documents.

In [ ]:
good_manifest = {
    "version": "1.0.0",
    "generated_at": "2026-03-11T12:00:00Z",
    "sources": [],
}

bad_manifest = {
    "version": "1.0.0",
    "generated_at": "not-a-datetime",
    "sources": [],
}

show_report("Good datasheet-source-manifest", validate_json_report(good_manifest, profile="datasheet-source-manifest"))
show_report("Bad datasheet-source-manifest", validate_json_report(bad_manifest, profile="datasheet-source-manifest"))


## 2. Canonical Record Validation

Record validation composes schema validation, optional reference validation, and semantic rules under one policy.

In [ ]:
good_dataset = load_example("datasets", "dataset-1f8r-6v2k-9p4m-3t7x.json")
bad_dataset = copy.deepcopy(good_dataset)
bad_dataset["dataset"]["url"] = "not-a-uri"
bad_dataset["dataset"]["dateModified"] = bad_dataset["dataset"]["dateCreated"] - 1

show_report(
    "Good canonical dataset (strict policy)",
    validate_record_report(good_dataset, source_root=EXAMPLES_ROOT, policy="strict"),
)
show_report(
    "Bad canonical dataset (strict policy)",
    validate_record_report(bad_dataset, policy="strict"),
)


## 3. Referential Integrity Validation

Reference validation checks that linked BattINFO IDs exist in a repository root and resolve to the correct record type.

In [ ]:
good_test = load_example("tests", "test-5p7v-2n8k-4m3t-6q9r.json")
bad_test = copy.deepcopy(good_test)
bad_test["test"]["dataset_ids"] = ["https://w3id.org/battinfo/dataset/zzzz-zzzz-zzzz-zzzz"]

show_report("Good test references", validate_references_report(good_test, EXAMPLES_ROOT))
show_report("Bad test references", validate_references_report(bad_test, EXAMPLES_ROOT))


## 4. Semantic Validation

Semantic checks catch problems that are structurally valid JSON but internally inconsistent.

In [ ]:
good_semantic = load_example("tests", "test-5p7v-2n8k-4m3t-6q9r.json")
bad_semantic = copy.deepcopy(good_semantic)
bad_semantic["test"]["short_id"] = "xxxxxx"

show_report("Good semantic example", validate_semantic_report(good_semantic, policy="strict"))
show_report("Bad semantic example", validate_semantic_report(bad_semantic, policy="strict"))


## 5. Publication Validation

Publication validation checks generated JSON-LD artifacts, including JSON-LD parseability and BattINFO publication-specific graph rules.

In [ ]:
workspace = Path(tempfile.mkdtemp(prefix="battinfo-validation-demo-"))
dataset_dir = workspace / "example-cell-001"
raw_file = dataset_dir / "data" / "run.csv"
raw_file.parent.mkdir(parents=True, exist_ok=True)
raw_file.write_text("time,voltage\n0,3.7\n", encoding="utf-8")

cell_type = CellType(
    id="https://w3id.org/battinfo/cell-type/1234-5678-9abc-def0",
    name="ExampleCell 21700-A",
    manufacturer="ExampleCell",
    model="21700-A",
    format="cylindrical",
    chemistry="NMC/Graphite",
    size_code="21700",
    nominal_properties={"nominal_voltage": {"value": 3.7, "unit": "V"}},
    source=ProvenanceInfo(type="manual", url="https://example.org/cells/21700-a", retrieved_at=1771804800),
)
cell = CellInstance(cell_type=cell_type, serial_number="example-cell-001")
test = Test(cell=cell, kind="cycle_life", protocol="cycling", status="completed")
dataset = Dataset(
    path=str(dataset_dir),
    cell=cell,
    test=test,
    name="ExampleCell measurements",
    description="Small validation demo dataset.",
)

publish_result = publish(
    cell_type=cell_type,
    cell_instance=cell,
    test=test,
    dataset=dataset,
)
publish_path = Path(publish_result["publish_path"])
good_publication = json.loads(publish_path.read_text(encoding="utf-8"))
bad_publication = copy.deepcopy(good_publication)

dataset_node = next(
    node
    for node in bad_publication["@graph"]
    if isinstance(node, dict) and str(node.get("@id", "")).startswith("https://w3id.org/battinfo/dataset/")
)
dataset_node["schema:distribution"][0]["schema:sha256"] = "1234"
dataset_node["schema:distribution"][0]["schema:contentUrl"] = "not-a-uri"

print("Workspace:", workspace)
show_report("Good publication artifact", validate_publication_report(good_publication, policy="publisher"))
show_report("Bad publication artifact", validate_publication_report(bad_publication, policy="publisher"))


## 6. Inspect Raw Issue Metadata

Downstream systems can consume issue codes, paths, severities, and validators directly.

In [ ]:
report = validate_record_report(bad_dataset, policy="strict")
issue = report.errors[0]
issue
